## 사전 준비: 라이브러리 및 데이터 로드


In [ ]:
# 필요 라이브러리 설치
!pip install statsmodels

# 코랩에서 한글 폰트 사용을 위한 설정
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# 런타임 다시 시작 안내
# 위 코드 실행 후 상단 메뉴에서 [런타임] > [런타임 다시 시작]을 눌러주세요.
# 런타임을 다시 시작해야 한글 폰트가 적용됩니다.

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-nanum is already the newest version (20200506-1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
/usr/share/fonts: caching, new cache contents: 0 fonts, 1 dirs
/usr/share/fonts/truetype: caching, new cache contents: 0 fonts, 3 dirs
/usr/share/fonts/truetype/humor-sans: caching, new cache contents: 1 fonts, 0 dirs
/usr/share/fonts/truetype/liberation: caching, new cache contents: 16 fonts, 0 dirs
/usr/share/fonts/truetype/nanum: caching, new cache contents: 12 fonts, 0 dirs
/usr/local/share/fonts: caching, new cache contents: 0 fonts, 0 dirs
/root/.local/share/fonts: skipping, no such directory
/root/.fonts: skipping, no such directory
/usr/share/fonts/truetype: skipping, looped directory detected
/usr/share/fonts/truetype/humor-sans: skipping, looped directory detected
/usr/share/fonts/truetype/liberation: skipping, looped directory detected
/usr/share/fonts/truetype/n

In [ ]:
# 런타임 다시 시작 후, 이 셀을 실행하여 라이브러리와 폰트를 로드합니다.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# 한글 폰트 설정
plt.rc('font', family='NanumGothic')
# plt.rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지

In [ ]:
# 실습용 'tips' 데이터셋을 불러옵니다.
tips = sns.load_dataset('tips')
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [ ]:
tips.info()
print(tips.describe())
print(tips['time'].unique())
print(tips['size'].unique())
print(tips['day'].unique())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   total_bill      244 non-null    float64 
 1   tip             244 non-null    float64 
 2   sex             244 non-null    category
 3   smoker          244 non-null    category
 4   day             244 non-null    category
 5   time            244 non-null    category
 6   size            244 non-null    int64   
 7   pred_male_prob  244 non-null    float64 
dtypes: category(4), float64(3), int64(1)
memory usage: 9.3 KB
       total_bill         tip        size  pred_male_prob
count  244.000000  244.000000  244.000000      244.000000
mean    19.785943    2.998279    2.569672        0.643443
std      8.902412    1.383638    0.951100        0.070023
min      3.070000    1.000000    1.000000        0.499114
25%     13.347500    2.000000    2.000000        0.591239
50%     17.795000    2.90

In [ ]:
tips.isnull().mean()

,0
total_bill,0.0
tip,0.0
sex,0.0
smoker,0.0
day,0.0
time,0.0
size,0.0




1.   데이터를 확인해보니 tip은 연속형 -> OLS (최소제곱법 선형회귀)
2.   sex, smoker, day, time은 범주형 -> 더미 필요
3. size는 연속형 -> 그대로 사용
4. 결측치 없음





### 📝 Statsmodels 사용법

1.  **데이터 분리 (X, y)**: 분석할 데이터를 **독립변수 `X`**(설명하는 변수들, DataFrame)와 **종속변수 `y`**(알고싶은 결과, Series)로 직접 나눕니다.
2.  **데이터 가공**:
    - **더미 변수 생성**: `pd.get_dummies()` 함수를 사용해 문자(범주형) 데이터를 0과 1로 이루어진 숫자 데이터로 직접 변환합니다.
    - **상수항(Intercept) 추가**: **가장 중요한 단계입니다.** 회귀식의 y절편(`b`ใน `y=ax+b`)을 계산하기 위해 `sm.add_constant(X)`를 사용하여 X 데이터에 상수항 컬럼을 **수동으로 추가**해야 합니다.
3.  **모델 학습**: `sm.OLS(y, X)` 또는 `sm.Logit(y, X)` 함수에 준비된 `y`와 `X` 데이터를 전달하여 모델을 학습시킵니다.

이제 이 3단계를 따라 모든 문제를 해결해 봅시다.


---


## 문제 1 (난이도: 하): 다중회귀분석의 첫걸음

> **🎯 목표:** 여러 개의 원인(독립변수)을 동시에 고려하여 결과를 예측하는 **다중회귀분석**을 파이썬 배열 방식으로 구현합니다.


### 💡 핵심 개념:

단순회귀가 하나의 원인(`X`)으로 결과를 설명했다면, **다중회귀**는 **여러 개의 원인(`X1, X2, ...`)**을 동시에 고려하여 결과를 더 정교하게 설명하는 방법입니다. 이때 각 원인의 영향력(계수)은 **'다른 원인들이 모두 동일하다고 통제했을 때'**의 순수한 영향력을 의미합니다.


### 📌 수행 과제:

1. `tips` 데이터셋에서 `tip`을 종속변수(y)로, `total_bill`과 `size`를 독립변수(X)로 분리하세요.
2. 독립변수(X)에 상수항을 추가하세요.
3. `sm.OLS`를 사용하여 다중회귀 모델을 학습시키고, `.summary()`로 결과를 확인하세요.
4. 결과표를 보고 `total_bill`과 `size`의 계수(coef)가 각각 무엇을 의미하는지 해석해 보세요.


### ✍️ 코드 작성:


파이썬 (numpy) 방식으로 다중회귀 돌려보기
1. X,Y 만들기
2. 더미 처리
3. 상수항 추가
4. OLS 적합 확인

In [ ]:
# 1. 여기에 데이터를 y와 X로 분리하는 코드를 작성하세요.
# 종속변수 y에 'tip' 컬럼을 할당하세요.
y = tips["tip"].values #팁 금액 컬럼을 numpy array로 변환

# 독립변수 X에 'total_bill'과 'size' 컬럼을 리스트로 묶어 할당하세요.
X = tips[["total_bill", "size"]] #총 계산금액과 인원수를 원인변수로 넣기(리스트로 데이터프레임형태 유지)

# 2. 여기에 상수항을 추가하는 코드를 작성하세요. -> 해당 방식에 맞게 위에 배열/API 방식으로 바꿈(import statsmodel.api as sm)
# sm.add_constant()를 사용해 X 데이터에 상수항을 추가합니다.
X_const = sm.add_constant(X)

# 3. 여기에 모델을 학습하고 결과를 출력하는 코드를 작성하세요.
# sm.OLS() 함수에 y와 X_const를 순서대로 넣어 모델을 만들고 학습시킵니다. (formula 방식이면 절편 자동 포함이고 " ~ +" 사용)
model_1 = sm.OLS(y, X_const).fit()
print(model_1.summary())

# 4. 결과 해석 (아래 주석에 직접 작성해 보세요)
# total_bill 계수의 의미: 다른 조건이 같을 때, 총 계산 금액 1달러 증가하면 팁은 평균적으로 0.09달러만큼 증가함
# size 계수의 의미: 다른 조건이 같을 때, 테이블 인원수 1명 증가 시 팁은 평균적으로 0.19달러만큼 증가
# 이 모델은 R-squared 값으로 보아 팁 변동의 약 46.8% 를 설명함

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.468
Model:                            OLS   Adj. R-squared:                  0.463
Method:                 Least Squares   F-statistic:                     105.9
Date:                Wed, 17 Dec 2025   Prob (F-statistic):           9.67e-34
Time:                        06:41:58   Log-Likelihood:                -347.99
No. Observations:                 244   AIC:                             702.0
Df Residuals:                     241   BIC:                             712.5
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.6689      0.194      3.455      0.0

### 🤔 생각해 볼 문제:

`X_const.head()`를 출력해서 `sm.add_constant()`가 실제로 X 데이터에 어떤 변화를 주었는지 직접 눈으로 확인해보세요.


In [ ]:
# 회귀식에서 보통 절편이 들어가는데 해당 코드로 모든 관측치에 값이 1인 const=1 컬럼을 추가해 회귀모형에서 절편을 계수형태로 추정하게 해줌.
X_const.head()

,const,total_bill,size
0,1.0,16.99,2
1,1.0,10.34,3
2,1.0,21.01,3
3,1.0,23.68,2
4,1.0,24.59,4


---


## 문제 2 (난이도: 하): 더미 변수 활용하기

> **🎯 목표:** '요일'과 같은 문자(범주형) 데이터를 분석에 활용하기 위해 **더미 변수**로 변환하는 방법을 배우고, 회귀분석에 적용합니다.


### 💡 핵심 개념:

**더미 변수**는 '성별', '요일'과 같은 문자(범주형) 데이터를 통계 모델에 사용하기 위해 **0과 1로 이루어진 숫자 변수**로 변환하는 기법입니다. Pandas의 `pd.get_dummies()` 함수를 사용하면 이 과정을 쉽게 처리할 수 있습니다. 이때, 여러 범주 중 하나는 **기준(reference)**이 되어 분석 오류를 막고 해석의 기준점 역할을 합니다.


### 📌 수행 과제:

1. `total_bill`과 '요일(`day`)'이 팁(`tip`)에 미치는 영향을 분석해봅시다.
2. `pd.get_dummies()`를 사용하여 'day' 컬럼을 더미 변수로 변환하세요. (단, `drop_first=True` 옵션 사용)
3. 기존 독립변수(`total_bill`)와 생성된 더미 변수를 합쳐 최종 X 데이터를 만드세요.
4. 상수항을 추가하고 회귀 모델을 학습시킨 후, 결과를 해석하세요.


### ✍️ 코드 작성:


In [ ]:
# 1. 기본 데이터를 종속변수와 독립변수로 분리
y = tips['tip']
X_base = tips[['total_bill']]

# 2. 여기에 'day' 컬럼으로 더미 변수를 생성하는 코드를 작성
# pd.get_dummies()를 사용하고, 기준 요일을 자동으로 제거하기 위해 drop_first=True 옵션지정
# x값 숫자로 통일해 모델 입력 오류 예방 (x 타입이 float, blool, uint 혼합이면 object로 인식해 에러남)
day_dummies = pd.get_dummies(tips['day'], prefix='day', drop_first=True).astype(int)

# 3. 여기에 기본 X 데이터와 더미 변수를 합치는 코드를 작성
# pd.concat을 사용하고, axis=1 옵션으로 옆으로 붙임
X = pd.concat([X_base, day_dummies], axis=1)


# 4. 상수항을 추가하고 모델을 학습시킨 후 결과를 출력
# 배열/API 방식으로 바꿈(import statsmodel.api as sm)- sm.add_constant()를 사용해 X 데이터에 상수항을 추가
# 기본 기준 확률(절편) - 배열API 상수항 추가 안하면 모델이 (0,0)에서만 시작해야해서 해석 왜곡됨.
X_const = sm.add_constant(X)
model_2 = sm.OLS(y, X_const).fit()
print(model_2.summary())

day_dummies.columns

# 5. 결과 해석
# total_bill 계수의 의미 : 다른 조건이 같을 때 총 계산금액 1달러 증가 시 팁은 평균적으로 0.1047 증가한다. (통계적으로 유의미함)
# 기준이 된 요일은 무엇인가?: 목요일
# 'day_Sat' 계수의 의미 : 다른 조건이 같을 때 목요일에 비해서 토요일은 팁이 평균적으로 0.0671만큼 낮다고 나왔으나 통계적으로 유의미하지 않음.
# 현재 요일이 팁 변동을 설명하는 부분에 있어서는 p밸류가 0.05보다 크므로 통계적으로 유의미한 결과가 아님
#  즉 요일은 팁 변동을 설명하기에 통계적으로 유용한 변수가 아닙니다.

                            OLS Regression Results                            
Dep. Variable:                    tip   R-squared:                       0.459
Model:                            OLS   Adj. R-squared:                  0.450
Method:                 Least Squares   F-statistic:                     50.67
Date:                Wed, 17 Dec 2025   Prob (F-statistic):           7.52e-31
Time:                        06:41:58   Log-Likelihood:                -350.03
No. Observations:                 244   AIC:                             710.1
Df Residuals:                     239   BIC:                             727.5
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.9205      0.186      4.943      0.0

Index(['day_Fri', 'day_Sat', 'day_Sun'], dtype='object')

### 🤔 생각해 볼 문제:

`drop_first=True` 옵션을 빼고 `False`로 설정한 뒤 모델을 다시 학습시켜보세요. 결과가 어떻게 달라지나요? (힌트: `statsmodels`가 다중공선성에 대한 경고 메시지를 보여줄 수 있습니다.)


모델 일단 돌아감. R스퀘어 값도 동일, total_bill계수도 동일
하지만 맨 하단에 경고 메시지 확인하는 것이 중요함!!

In [ ]:
# 1. 기본 데이터를 X와 Y로 분리합니다.
y = tips['tip']
X_base = tips[['total_bill']]

# 2. 문자 범주형 컬럼인'day'를 통계모델이 읽을 수 있는 형태로 더미 변수(0,1) 생성
# pd.get_dummies()를 사용하고, 기준 요일을 하나 자동으로 제거하기 위해 drop_first=True 옵션과 x값 숫자로 통일
 #(x 타입이 float, blool, uint 혼합이면 object로 인식해 에러남)
day_dummies = pd.get_dummies(tips['day'], prefix='day', drop_first=False).astype(int)

# 3. 독립변수 결합 : 기본 X 데이터와 더미 변수를 합침
# pd.concat을 사용하고, axis=1 옵션으로 옆으로 붙입니다.
X = pd.concat([X_base, day_dummies], axis=1)


# 4. 상수항을 추가하고 선형회귀 모델을 학습시킨 후 결과 출력
X_const = sm.add_constant(X)
model_2 = sm.OLS(y, X_const).fit()
print(model_2.summary())

day_dummies.columns


                            OLS Regression Results                            
Dep. Variable:                    tip   R-squared:                       0.459
Model:                            OLS   Adj. R-squared:                  0.450
Method:                 Least Squares   F-statistic:                     50.67
Date:                Wed, 17 Dec 2025   Prob (F-statistic):           7.52e-31
Time:                        06:41:58   Log-Likelihood:                -350.03
No. Observations:                 244   AIC:                             710.1
Df Residuals:                     239   BIC:                             727.5
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.7455      0.131      5.677      0.0

Index(['day_Thur', 'day_Fri', 'day_Sat', 'day_Sun'], dtype='object')

(1) 표준오차는 오차의 공분산 행렬이 올바르게 지정되었다고 가정한다. (즉, 등분산성, 독립성등의 OLS 가정이 맞다고 전제한 표준오차란 이야기임)
**(2)가장 작은 고유값이 2.35*10-30 이다. 이는 강한 다중공선성 문제가 있거나 설계행렬이 특이할 수 있음을 말한다.** - 고유값이 거의 0이란 뜻은 변수간 정보가 겹친다는 의미이며, 설계행렬에서 해는 존재하나 매우 불안정하여 작은 변화에도 계수가 크게 흔들릴 수 있다는 뜻임. : 즉 정보가 겹쳐서 실제 어떤 변수가 영향을 주는지 명확히 구분하기 어렵다는 이야기임. 따라서 이 데이터상에서 개별 변수의 p값과 계수 해석은 신뢰하기 어렵다.

## 문제 3 (난이도: 중): 상호작용 효과 분석하기

> **🎯 목표:** 한 변수의 효과가 다른 변수의 상황에 따라 달라지는 **상호작용 효과**를 직접 변수를 곱하여 만들고, 그 의미를 해석합니다.


### 💡 핵심 개념:

**상호작용**은 한 변수의 효과가 다른 변수의 수준에 따라 달라지는 현상을 말합니다. 예를 들어, "식사 금액이 팁에 미치는 영향은 흡연 여부에 따라 다르다"와 같은 가설을 검증할 때 사용됩니다. 배열 방식에서는 상호작용 항을 **두 변수의 곱**으로 직접 만들어서 모델에 추가합니다.


### 📌 수행 과제:

1. "식사 총액(`total_bill`)이 팁에 미치는 영향은 흡연자(`smoker`) 그룹과 비흡연자 그룹에서 다를 것이다"라는 가설을 검증해봅시다.
2. `smoker` 변수를 더미 변수(`smoker_Yes`)로 만드세요. ('Yes' = 1, 'No' = 0)
3. `total_bill`과 `smoker_Yes`를 곱하여 상호작용 항(`bill_x_smoker`)을 만드세요.
4. `total_bill`, `smoker_Yes`, `bill_x_smoker`를 모두 포함하여 회귀 모델을 학습시키고 결과를 해석하세요.


### ✍️ 코드 작성:


In [ ]:
# 1. 기본 데이터를 준비합니다.
y = tips['tip']
X = tips[['total_bill']]

# 2. 여기에 'smoker' 더미 변수를 X에 추가하는 코드를 작성하세요.
X['smoker_Yes'] = pd.get_dummies(tips['smoker'], drop_first=True, dtype=int)

# 3. 여기에 상호작용 항을 직접 계산하여 X에 추가하는 코드를 작성하세요.
# 'total_bill' 컬럼과 'smoker_Yes' 컬럼을 곱합니다.
X['bill_x_smoker'] = X['total_bill'] * X['smoker_Yes']

# 4. 상수항을 추가하고 모델을 학습시킨 후 결과를 출력하세요.
X_const = sm.add_constant(X)
model_3 = sm.OLS(y, X_const).fit()
print(model_3.summary())

# 5. 결과 해석 (아래 주석에 직접 작성해 보세요)
# 상호작용 항(bill_x_smoker)의 p-value는 유의미한가?: 피밸류가 0.05 이하이므로 통계적으로 유의함.
# 즉,식사 총액이 팁에 미치는 영향은 흡연자와 비흡연자 간에 유의한 차이가 있다.
# 상호작용 항 계수의 의미:
# total_bill:  비흡연자 기준 계산 금액이 1증가할 때 평균적으로 0.069 증가
# smoker_yes: 계산 금액이 0일 때 흡연자는 비흡연자보다 팁이 평균 1.2 만큼 낮음
# bill_x_smoker : 흡연자일 때 식사총액이 팁에 미치는 효과가 비흡연자보다 0.0676만큼 더 커진다 (흡연자는 계산금액 증가에 더 민감하게 팁을 늘림))
# R 스퀘어도 상호작용 포함 시키니 설명력이 증가했음. (단순회귀보다 해석력이 좋아진 모델임)

                            OLS Regression Results                            
Dep. Variable:                    tip   R-squared:                       0.506
Model:                            OLS   Adj. R-squared:                  0.500
Method:                 Least Squares   F-statistic:                     81.95
Date:                Wed, 17 Dec 2025   Prob (F-statistic):           1.56e-36
Time:                        06:41:58   Log-Likelihood:                -338.91
No. Observations:                 244   AIC:                             685.8
Df Residuals:                     240   BIC:                             699.8
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             1.5643      0.238      6.570

/tmp/ipython-input-3464528325.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['smoker_Yes'] = pd.get_dummies(tips['smoker'], drop_first=True, dtype=int)
/tmp/ipython-input-3464528325.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['bill_x_smoker'] = X['total_bill'] * X['smoker_Yes']


### 🤔 생각해 볼 문제:

위 모델에서 비흡연자(smoker_Yes=0)의 `total_bill` 기울기는 얼마일까요? 그리고 흡연자(smoker_Yes=1)의 `total_bill` 기울기는 얼마일까요? (힌트: 흡연자의 기울기 = 기본 `total_bill` 기울기 + 상호작용 항 기울기)


	•	total_bill 기울기 = 0.0696
	•	흡연자의 경우, bill_x_smoker 기울기 ->  기본 기울기에 + 0.0676 더 해서 0.1372임.
	즉, 상호작용 항이 기울기를 바꾸는 스위치처럼 작용함.  그리고 더미변수 0값이 기준 그룹이 됨.

---


## 문제 4 (난이도: 중): 로지스틱 회귀분석

> **🎯 목표:** 결과가 '예/아니오' 같은 범주형일 때 사용하는 **로지스틱 회귀분석**을 구현하고, 그 결과를 **오즈비(Odds Ratio)**로 해석하는 방법을 배웁니다.


### 💡 핵심 개념:

**로지스틱 회귀분석**은 결과가 '예/아니오', '성공/실패'처럼 **두 가지 중 하나**일 때 사용하는 분석 방법입니다. 어떤 조건에서 특정 결과가 나타날 **'확률'**을 예측합니다. 회귀 계수는 그대로 해석하기 어렵고, `np.exp(계수)`를 계산한 **오즈비(Odds Ratio)**로 해석해야 합니다. 오즈비가 1보다 크면 확률 증가, 1보다 작으면 확률 감소를 의미합니다.


### 📌 수행 과제:

1. 식사 총액(`total_bill`)과 팁(`tip`) 금액을 보고, 그 고객이 '남성'일 확률을 예측하는 로지스틱 회귀 모델을 만들어봅시다.
2. 종속변수 `y`를 'sex'가 'Male'이면 1, 아니면 0이 되도록 변환하세요.
3. `sm.Logit` 함수를 사용하여 모델을 학습시키고, 결과를 확인하세요.
4. `np.exp()`를 이용해 계수를 오즈비로 변환하고, 그 의미를 해석하세요.


### ✍️ 코드 작성:


In [ ]:
# 1. 여기에 종속변수 y를 0과 1로 변환하는 코드를 작성하세요.
# 'sex'가 'Male'이면 1, 아니면 0
y = tips['sex'].apply(lambda x: 1 if x == 'Male' else 0)

# 2. 독립변수 X를 준비하고 상수항을 추가하세요.
X = tips[['total_bill', 'tip']]
X_const = sm.add_constant(X) #기본 기준 확률(절편) - 상수항 추가 안하면 모델이 (0,0)에서만 시작해야해서 해석 왜곡됨.

# 3. 여기에 로지스틱 회귀 모델을 학습시키는 코드를 작성하세요.
# sm.Logit() 함수를 사용합니다.
model_4 = sm.Logit(y, X_const).fit() #최대우도추정(MLE)으로 계수 추정함. 즉 확률변화가 아닌 로그오즈 변화가 결과값이 됨.
print(model_4.summary())

# 4. 여기에 오즈비를 계산하고 출력하는 코드를 작성하세요.
odds_ratios = np.exp(model_4.params)  #로지스틱 회귀계수 = 로그오즈, exp(계수)= 오즈비 : 사람에게 해석가능한 단위로 출력
print("\n--- 오즈비 (Odds Ratios) ---")
print(odds_ratios)

# 5. 결과 해석 (아래 주석에 직접 작성해 보세요)
# total_bill 오즈비의 의미: 식사 총액 1달러 증가 시 그 손님이 남성일 오즈는 평균적으로 1.04배만큼 증가한다 (1보다 크므로 확률 증가) 하지만 통계적으로 유의하다고 볼 수 없다.
# tip 오즈비의 의미: 팁 1달러 증가 시 그 손님이 남성일 오즈는 평균적으로 0.97배 만큼 감소한다. (1보다 작으므로 확률 감소) -> 여성일 가능성이 아주 약간 증가. 하지만 통계적으로 유의하다고 볼 수 없다.
#결론은 통계적으로 식사 금액, 팁 금액만 보고 성별을 설명하기란 어렵다.
#로지스틱 회귀에서는 확률과 우도(likehood)기반으로 설명력을 이야기하므로 Pseudo R-squ.값(0~1)이 0.017은 설명력이 매우 낮은 수준으로 판단함. (실무에선 0.2만 넘어도 쓸만하다고 함. 0.4 정도는 매우 강력)해당 변수는 성별 설명에 적합치 않음.
#LLR-p-value는 0.06으로 모델전체의 유의성을 나타내는데, 0.05 이상이라 귀무가설(이 모델이 의미없다)을 기각 못하므로 모델 전체가 통계적으로 유의하다 보기 어려움.

Optimization terminated successfully.
         Current function value: 0.640304
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                    sex   No. Observations:                  244
Model:                          Logit   Df Residuals:                      241
Method:                           MLE   Df Model:                            2
Date:                Wed, 17 Dec 2025   Pseudo R-squ.:                 0.01705
Time:                        06:57:07   Log-Likelihood:                -156.23
converged:                       True   LL-Null:                       -158.94
Covariance Type:            nonrobust   LLR p-value:                   0.06651
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0979      0.363     -0.270      0.787      -0.808       0.613
total_bill     0.0400      0.

### 🤔 생각해 볼 문제:

`model_4.predict(X_const)` 코드를 실행하면 각 손님에 대해 '남성일 확률'을 예측해줍니다. 이 예측 확률이 가장 높은 상위 5명의 손님 데이터(total_bill, tip)는 어떤 특징을 가지고 있는지 확인해보세요.


In [ ]:
# 1. 남성일 확률 예측
tips['pred_male_prob'] = model_4.predict(X_const)

# 2. 남성일 확률 기준으로 내림차순 정렬
top5 = tips.sort_values(by='pred_male_prob', ascending=False).head(5)

# 3. 필요한 변수만 확인
top5[['total_bill', 'tip', 'pred_male_prob']]

,total_bill,tip,pred_male_prob
156,48.17,5.00,0.843590
170,50.81,10.00,0.838758
59,48.27,6.73,0.837558
182,45.35,3.50,0.834105
102,44.30,2.50,0.832212


남성일 확률이 높은 상위 5명의 경우는 전체 계산금액이 큰 편에 속하고, 단 팁은 들쭉날쭉한 특성을 갖는다. (토탈빌이 남성 오즈를 증가시키고 팁은 감소시키는 방향이었던 로지스틱 회귀결과에 준하는 결과임) 모델 내부 기준에서 상대적 확률이 0.83~0.84로 높게 예측해주고 있지만, 사실 모델 신뢰도가 유의적이지 않았으므로 본 내용도 신뢰하기 어려움.

---


## 문제 5 (난이도: 중): 최고의 모델 찾기: AIC

> **🎯 목표:** 여러 모델 중 어떤 모델이 가장 좋은지 객관적인 점수로 평가하는 **AIC**의 개념을 이해하고, 이를 이용해 최적의 모델을 선택합니다.


### 💡 핵심 개념:

**AIC (Akaike Information Criterion)**는 여러 모델 중 어떤 모델이 가장 좋은지 평가하는 '점수'입니다. AIC는 **모델의 설명력(얼마나 잘 맞추는가)**은 높이고, **모델의 복잡성(얼마나 많은 변수를 썼는가)**에는 벌점을 줍니다. 따라서 **AIC 점수는 낮을수록 더 좋은(효율적인) 모델**이라고 평가합니다.


### 📌 수행 과제:

1. 팁(`tip`) 금액을 예측하기 위한 세 가지 다른 모델을 만듭니다.
   - 모델 A: `total_bill`만 사용
   - 모델 B: `total_bill`, `size` 사용
   - 모델 C: `total_bill`, `size`, `day`(더미) 사용
2. 각 모델의 `.aic` 속성을 확인하여 AIC 점수를 비교하세요.
3. 어떤 모델이 가장 좋은 모델인지 선택하고, 그 이유를 설명하세요.


### ✍️ 코드 작성:


In [ ]:
y = tips['tip']

# --- 모델 A: total_bill만 사용 ---
X_A = tips[['total_bill']]
X_A_const = sm.add_constant(X_A)
model_A = sm.OLS(y, X_A_const).fit()
print("Model A AIC:", model_A.aic)

# --- 모델 B: total_bill과 size 사용 ---
# 1. 여기에 모델 B의 X 데이터를 준비하고, 학습 후 AIC를 출력하는 코드를 작성하세요.
X_B = tips[['total_bill', 'size']]
X_B_const = sm.add_constant(X_B)
model_B = sm.OLS(y, X_B_const).fit()
print("Model B AIC:", model_B.aic)

# --- 모델 C: total_bill, size, day 사용 ---
# 2. 여기에 모델 C의 X 데이터를 준비하고, 학습 후 AIC를 출력하는 코드를 작성하세요.
# (힌트: 문제 2에서 X 데이터를 만들었던 과정을 참고하세요.)
day_dummies = pd.get_dummies(tips['day'], prefix='day', drop_first=True, dtype=int)

X_C = pd.concat([X_B, day_dummies], axis=1)
X_C_const = sm.add_constant(X_C)
model_C = sm.OLS(y, X_C_const).fit()
print("Model C AIC:", model_C.aic)


# 3. 결과 해석 (아래 주석에 직접 작성해 보세요)
# 가장 좋은 모델(AIC가 가장 낮은 모델)은?:B
# 그렇게 생각하는 이유: AIC는 점수가 낮을수록 좋은 모델이다.

Model A AIC: 705.0761662637547
Model B AIC: 701.9702091086953
Model C AIC: 707.3838897060045


### 🤔 생각해 볼 문제:

AIC는 모델의 복잡도(사용한 변수의 개수)에 벌점을 줍니다. 모델 B는 모델 A보다 변수가 하나 더 많음에도 AIC가 낮아졌습니다. 이는 무엇을 의미할까요?

AIC는 잘 맞추는 모델 중에서도 쓸데없이 복잡하지 않은 모델을 고르는 점수다.  값이 낮을 수록 더 좋은 모델이다.
B가 A보다 변수가 늘어났음에도 불구하고 해당 점수가 좋아진 것은 복잡도가 증가해 벌점을 받았더라도, 동시에 size변수가 보조설명변수로 작용하면서 잘 맞추는 쪽으로 개선된 것이 더 크게 작용했기 때문이다.

C 같은 경우 더미 추가하면서 복잡도 크게 증가했지만 day는 tip을 설명하는데 추가 정보가 거의 없음. 그 결과 쓸데없이 복잡해지기만 했다.
